In [2]:
import numpy as np

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

---

In [4]:
# Forward pass - Python
def forwardPy(inp, W, bias):
    z = 0
    for i in range(len(inp)):
        z += (inp[i] * W[i])
    z += bias
    a = sigmoid(z)
    return z, a

In [5]:
# Forward pass - Numpy
def forwardNp(inp, W, bias):
    z = np.dot(inp, W) + bias
    a = sigmoid(z)
    return z, a

In [180]:
# Test forward pass
num_feats = 2
W = np.random.randn(num_feats)
bias = np.random.randn()
inp = np.random.randn(num_feats)

assert forwardPy(inp, W, bias) == forwardNp(inp, W, bias)

---

In [181]:
# MSE loss - Python - works only with lists/tuples
def MSELossPy(preds, targs):
    loss = 0
    len_preds = len(preds)
    for i in range(len_preds):
        loss += (preds[i] - targs[i]) ** 2
    loss /= len_preds
    return loss

assert MSELossPy([0, 0], [0, 1]) == 0.5
assert MSELossPy([1, 1], [1, 1]) == 0
assert MSELossPy([0, 0], [1, 1]) == 1

In [72]:
# MSE loss - Numpy - works with numpy arrays, and scalars
def MSELossNp(preds, targs):
    return np.mean((preds - targs) ** 2)

assert MSELossNp(np.array([0, 0]), np.array([0, 1])) == 0.5
assert MSELossNp(np.array([1, 1]), np.array([1, 1])) == 0
assert MSELossNp(np.array([0, 0]), np.array([1, 1])) == 1
assert MSELossNp(0, 0) == 0
assert MSELossNp(0, 1) == 1


---

In [6]:
# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # (4, 2)
y = np.array([0, 1, 1, 0]) # (4,)
num_samples = len(X) # 4

print(X[0])
print(X[:, 0])
print(X[0, :])

[0 0]
[0 0 1 1]
[0 0]


---

In [7]:
# Model params - One neuron
num_features = X.shape[1]
W = np.random.randn(num_features)
bias = np.random.randn()

print(W)
print(bias)

[1.27817139 0.86321874]
-1.0807339397802225


In [185]:
z, a = forwardNp(X, W, bias)
loss = MSELossNp(a, y)

print("z:", z)
print("a:", a)
print("y:", y)
print("loss:", loss)

z: [ 0.19308752  0.50492439 -0.46788986 -0.15605299]
a: [0.54812246 0.62361588 0.38511581 0.46106573]
y: [0 1 1 0]
loss: 0.2581918545232616


In [186]:
def backprop(X, y, a):
    dL_da = 2 * (a - y) # (4,)
    da_dz = a * (1 - a) # (4,)
    dz_dW = X # (4, 2)
    dL_dz = dL_da * da_dz # (4,)

    num_samples = len(X)
    dL_dW = np.dot(dz_dW.T, dL_dz) / num_samples # (2, 4) @ (4,) -> (2,)

    dL_dB = np.mean(dL_dz) # average over all samples
    return dL_dW, dL_dB

In [187]:
# Train the neuron
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    z, a = forwardNp(X, W, bias)

    # loss
    loss = MSELossNp(a, y)

    # backward
    gradW, gradB = backprop(X, y, a)

    # update
    W = W - lr * gradW
    bias = bias - lr * gradB

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.2581918545232616
loss at epoch 500: 0.2561814994805921
loss at epoch 1000: 0.2546833266789315
loss at epoch 1500: 0.25353387397607746
loss at epoch 2000: 0.2526563644825723
loss at epoch 2500: 0.2519920388693464
loss at epoch 3000: 0.2514923593112212
loss at epoch 3500: 0.2511181236267235
loss at epoch 4000: 0.25083852789274597
loss at epoch 4500: 0.2506298595197188


In [188]:
# Test the neuron
_, a = forwardNp([0,0], W, bias)
print(a)
_, a = forwardNp([0,1], W, bias)
print(a)
_, a = forwardNp([1,0], W, bias)
print(a)
_, a = forwardNp([1,1], W, bias)
print(a)

0.5178559695464336
0.5284637194098327
0.4759402249900127
0.4865558313799513


---

In [124]:
# Model params - Two neurons
num_features = X.shape[1]
W1 = np.random.randn(num_features)
b1 = np.random.randn()
W2 = np.random.randn()
b2 = np.random.randn()

print(f"W1: {W1}\nW2: {W2}")
print(f"b1: {b1}\nb2: {b2}")

W1: [0.79040768 0.73532508]
W2: 0.12189530270509348
b1: -0.30713473123741786
b2: -0.22175485362447783


In [125]:
# Forward pass
def forward_two_neurons(W1, b1, W2, b2, X):
    _, a1 = forwardNp(X, W1, b1)
    _, a2 = forwardNp(a1, W2, b2)
    return a1, a2

In [126]:
# Test forward pass
a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)
print(f"a1: {a1}\na2: {a2}")

a1: [0.42381427 0.60544146 0.61852043 0.77181673]
a2: [0.45757876 0.46307873 0.46347515 0.46812478]


In [181]:
# Backward pass
def backward_two_neurons(x, y, a1, a2, w2):
    num_samples = x.shape[0]   # () -> 4

    dl_da2 = 2 * (a2 - y)       # (4,)
    da2_dz2 = a2 * (1 - a2)     # (4, )
    dz2_dw2 = a1                # (4,)
    dz2_db2 = 1                 # ()

    dz2_da1 = w2                # ()
    da1_dz1 = a1 * (1 - a1)     # (4,)
    dz1_dw1 = x                 # (4, 2)
    dz1_db1 = 1                 # ()

    dl_dz2 = dl_da2 * da2_dz2   # (4,)

    dl_dw2 = np.dot(dl_dz2, dz2_dw2) / num_samples  # () which matches W2's shape
    dl_db2 = np.mean(dl_dz2 * dz2_db2)              # () which matches b2's shape

    dl_dz1 = dl_dz2 * dz2_da1 * da1_dz1                 # (4,) -> the gradient sent back to neuron 1

    dl_dw1 = np.dot(dz1_dw1.T, dl_dz1) / num_samples    # (2, 4) @ (4,) -> (2,) which matches W1's shape
    dl_db1 = np.mean(dl_dz1 * dz1_db1)                  # () which matches b1's shape

    return dl_dw2, dl_db2, dl_dw1, dl_db1

In [182]:
# Test backward pass
backward_two_neurons(X, y, a1, a2, W2)

ValueError: operands could not be broadcast together with shapes (4,) (2,) 

In [132]:
# Train 2 neurons
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)

    # loss
    loss = MSELossNp(a2, y)

    # backward
    dl_dw2, dl_db2, dl_dw1, dl_db1 = backward_two_neurons(X, y, a1, a2, W2)

    # update weights
    W1 = W1 - lr * dl_dw1
    b1 = b1 - lr * dl_db1

    W2 = W2 - lr * dl_dw2
    b2 = b2 - lr * dl_db2

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.2495115312059341
loss at epoch 500: 0.24948675961160688
loss at epoch 1000: 0.24946052602435836
loss at epoch 1500: 0.2494327280770306
loss at epoch 2000: 0.249403255818815
loss at epoch 2500: 0.24937199111849545
loss at epoch 3000: 0.2493388070537063
loss at epoch 3500: 0.24930356727054423
loss at epoch 4000: 0.24926612531031053
loss at epoch 4500: 0.24922632390331617


In [133]:
# Test the 2 neurons
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,1])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,1])
print(a2)

0.486742148233203
0.502884434844754
0.5037678995497576
0.5161858923107494


---

In [250]:
# Model params - MLP
num_features = X.shape[1]
neurons_in_hidden_layer = 2
W1 = np.random.randn(neurons_in_hidden_layer, num_features)
b1 = np.random.randn(neurons_in_hidden_layer)
W2 = np.random.randn(neurons_in_hidden_layer)
b2 = np.random.randn()

print(f"W1: {W1}\nW2: {W2}")
print(f"b1: {b1}\nb2: {b2}")

W1: [[ 0.9206485   0.93386912]
 [-0.63097872 -1.06000627]]
W2: [ 1.07014811 -0.42573095]
b1: [ 1.95759242 -0.3300133 ]
b2: 1.4557204329491662


In [251]:
a1, a2 = forward_two_neurons(W1, b1, W2, b2, X) # works for MLP too
print(f"a1: {a1}\na2: {a2}")

a1: [[0.87627216 0.41823739]
 [0.79027995 0.19940463]
 [0.94676027 0.64653796]
 [0.90441412 0.38789944]]
a2: [0.90162437 0.90172539 0.89967719 0.90537579]


In [254]:
# Backward pass - MLP
def backward_mlp(x, y, a1, a2, w2):
    num_samples = x.shape[0]   # () -> 4

    dl_da2 = 2 * (a2 - y)       # (4,)
    da2_dz2 = a2 * (1 - a2)     # (4, )
    dz2_dw2 = a1                # (4, 2)
    dz2_db2 = 1                 # ()

    dz2_da1 = w2                # (2,)
    da1_dz1 = a1 * (1 - a1)     # (4, 2)
    dz1_dw1 = x                 # (4, 2)
    dz1_db1 = 1                 # ()

    dl_dz2 = dl_da2 * da2_dz2   # (4,)

    dl_dw2 = np.dot(dl_dz2, dz2_dw2) / num_samples      # (2,) which matches W2's shape
    dl_db2 = np.mean(dl_dz2 * dz2_db2)                  # () which matches b2's shape

    dl_dz1 = (dl_dz2[:, None] * dz2_da1) * da1_dz1      # (4, 1) outer prod (2,) * (4, 2) -> (4, 2) the gradient sent back to layer 1
    # TODO: try the dot product version

    dl_dw1 = np.dot(dz1_dw1.T, dl_dz1) / num_samples    # (2, 4) @ (4, 2) -> (2, 2) which matches W1's shape
    dl_db1 = np.mean(dl_dz1 * dz1_db1, axis=0)          # (2,) which matches b1's shape

    return dl_dw2, dl_db2, dl_dw1, dl_db1

In [255]:
backward_mlp(X, y, a1, a2, W2)

(4,)
(4, 1)
(2,)


(array([0.06238605, 0.02797171]),
 np.float64(0.06988620356352046),
 array([[ 0.00334364, -0.00347971],
        [ 0.00281555, -0.00362425]]),
 array([ 0.0072107 , -0.00732579]))

In [238]:
# Train MLP
epochs = 100000
lr = 0.5
for epoch in range(epochs):
    # forward
    a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)

    # loss
    loss = MSELossNp(a2, y)

    # backward
    dl_dw2, dl_db2, dl_dw1, dl_db1 = backward_mlp(X, y, a1, a2, W2)

    # update weights
    W1 = W1 - lr * dl_dw1
    b1 = b1 - lr * dl_db1

    W2 = W2 - lr * dl_dw2
    b2 = b2 - lr * dl_db2

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.32993621866354034
loss at epoch 500: 0.24498629957370832
loss at epoch 1000: 0.2114529562522362
loss at epoch 1500: 0.16642953887434703
loss at epoch 2000: 0.1429951022835405
loss at epoch 2500: 0.10371765288032642
loss at epoch 3000: 0.016424251461821037
loss at epoch 3500: 0.007467146595400945
loss at epoch 4000: 0.0046697125601650085
loss at epoch 4500: 0.0033549556358370845
loss at epoch 5000: 0.0026016809846740405
loss at epoch 5500: 0.002116939512393657
loss at epoch 6000: 0.0017802796330744805
loss at epoch 6500: 0.0015335182046861949
loss at epoch 7000: 0.0013452516452833927
loss at epoch 7500: 0.0011970945061694936
loss at epoch 8000: 0.001077591785944373
loss at epoch 8500: 0.0009792465398467437
loss at epoch 9000: 0.0008969525222697656
loss at epoch 9500: 0.0008271150630860551
loss at epoch 10000: 0.0007671323111749023
loss at epoch 10500: 0.0007150758506543411
loss at epoch 11000: 0.0006694868442050116
loss at epoch 11500: 0.0006292418139804112
loss at ep

In [239]:
# Test MLP
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,1])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,1])
print(a2)

0.008004688462888007
0.9931712436508677
0.9932677362179031
0.006952361624552289
